# DTD Jupyter Calculation Module

This notebook combines confirmed `vanke` history with unconfirmed Temporary data and calculates one DTD value using observations dated on or before the target date only.

- Confirmed and Temporary data use the same schema; confirmed data take priority when company/date pairs overlap.
- New Temporary trading dates extend the history; missing financial-statement fields are forward-filled from the latest earlier record.
- The target date must exist explicitly; the calculation never silently substitutes an earlier trading day.
- Defaults: a one-year rolling window, 250 trading days per year, at least 50 valid observations, and δ = 0.50.
- This is a reconstruction consistent with the supplied NUS-CRI methodology; it does not claim to reproduce proprietary production parameters exactly.

## 1 Input parameters

For daily use, update only this cell:

1. `CONFIRMED_FILE`: the confirmed-history file.
2. `TEMPORARY_FILE`: the unconfirmed file; use `None` when there is none. A pandas DataFrame is also accepted.
3. `TARGET_DATE`: the trading date to calculate.
4. `COMPANY`: set this only when the file contains multiple companies.

In [ ]:
from pathlib import Path

PROJECT_DIR = Path.cwd().resolve()
CONFIRMED_FILE = PROJECT_DIR / "vanke.xlsx"
TEMPORARY_FILE = None  # For example: PROJECT_DIR / "vanke_dtd_temporary_data.xlsx"
TARGET_DATE = "2025-12-12"
COMPANY = None         # Keep None for one company; use a Comp_no such as 5338 for multiple companies.
INPUT_SHEET = "Input" # Input worksheet name in the Excel files.

## 2 Complete calculation module

In [ ]:
from __future__ import annotations

from dataclasses import asdict, dataclass
from pathlib import Path
from typing import Optional, Union

import numpy as np
import pandas as pd
from scipy.optimize import brentq, minimize_scalar
from scipy.stats import norm


COLS = {
    "company": "Comp_no",
    "date": "Date",
    "equity": "CUR_MKT_CAP(HKD)",
    "cl": "BS_CUR_LIAB(HKD)",
    "ltd": "BS_LT_BORROW(HKD)",
    "tl": "BS_TOT_LIAB2(HKD)",
    "assets": "BS_TOT_ASSET(HKD)",
    "rf": "Risk_Free_Rate",
}
REQUIRED_COLUMNS = [COLS[key] for key in ("date", "equity", "cl", "ltd", "tl", "assets", "rf")]
BS_COLUMNS = [COLS[key] for key in ("cl", "ltd", "tl", "assets")]
TableInput = Union[str, Path, pd.DataFrame]


@dataclass(frozen=True)
class DTDResult:
    date: pd.Timestamp
    dtd: float
    sigma: float
    asset_value: float
    default_point: float
    n_obs: int
    delta: float
    source_status: str

    def as_dict(self) -> dict:
        result = asdict(self)
        result["date"] = self.date.strftime("%Y-%m-%d")
        return result


def _parse_dates(series: pd.Series) -> pd.Series:
    """Accept Excel dates, YYYYMMDD integers, and conventional date strings."""
    text = series.astype(str).str.strip().str.replace(r"\.0$", "", regex=True)
    compact = text.str.fullmatch(r"\d{8}")
    parsed = pd.to_datetime(text, errors="coerce")
    if compact.any():
        parsed.loc[compact] = pd.to_datetime(text.loc[compact], format="%Y%m%d", errors="coerce")
    return parsed.dt.normalize()


def _rate_to_decimal(value: float) -> float:
    """Interpret 2.46 as 2.46%; retain 0.0246 as a decimal rate."""
    value = float(value)
    return value / 100.0 if abs(value) > 1.0 else value


def _read_table(source: Optional[TableInput], sheet_name: str = "Input") -> pd.DataFrame:
    if source is None:
        return pd.DataFrame()
    if isinstance(source, pd.DataFrame):
        return source.copy()
    path = Path(source)
    if not path.exists():
        raise FileNotFoundError(f"Input file was not found: {path}")
    suffix = path.suffix.lower()
    if suffix in {".xlsx", ".xls", ".xlsm"}:
        return pd.read_excel(path, sheet_name=sheet_name)
    if suffix == ".csv":
        return pd.read_csv(path)
    if suffix == ".parquet":
        return pd.read_parquet(path)
    raise ValueError(f"Unsupported file format: {suffix}")


def _validate_schema(frame: pd.DataFrame, label: str) -> None:
    missing = [column for column in REQUIRED_COLUMNS if column not in frame.columns]
    if missing:
        raise ValueError(f"{label} is missing required columns: {missing}")


def combine_confirmed_and_temporary(
    confirmed: TableInput,
    temporary: Optional[TableInput] = None,
    *,
    confirmed_sheet: str = "Input",
    temporary_sheet: str = "Input",
    company: Optional[object] = None,
) -> pd.DataFrame:
    """Combine data and return clean history; confirmed data override Temporary data for duplicate dates."""
    confirmed_df = _read_table(confirmed, confirmed_sheet)
    temporary_df = _read_table(temporary, temporary_sheet)
    _validate_schema(confirmed_df, "confirmed")
    if not temporary_df.empty:
        _validate_schema(temporary_df, "temporary")

    confirmed_df = confirmed_df.copy()
    confirmed_df["_source_status"] = "confirmed"

    frames = [confirmed_df]
    if not temporary_df.empty:
        temporary_df = temporary_df.copy()
        temporary_df["_source_status"] = "temporary"
        # When Temporary data lack Comp_no and confirmed history contains one company, fill it automatically.
        if COLS["company"] not in temporary_df.columns and COLS["company"] in confirmed_df.columns:
            companies = confirmed_df[COLS["company"]].dropna().unique()
            if len(companies) == 1:
                temporary_df[COLS["company"]] = companies[0]
        frames.append(temporary_df)

    data = pd.concat(frames, ignore_index=True, sort=False)
    data[COLS["date"]] = _parse_dates(data[COLS["date"]])
    if data[COLS["date"]].isna().any():
        bad_rows = data.index[data[COLS["date"]].isna()].tolist()[:10]
        raise ValueError(f"Unparseable dates found; example row indexes: {bad_rows}")

    has_company = COLS["company"] in data.columns
    if company is not None:
        if not has_company:
            raise ValueError("A company was specified, but the input has no Comp_no column.")
        data = data[data[COLS["company"]] == company].copy()
        if data.empty:
            raise ValueError(f"Comp_no was not found in the input: {company}")
    elif has_company:
        companies = data[COLS["company"]].dropna().unique()
        if len(companies) > 1:
            raise ValueError(f"The file contains multiple companies {companies.tolist()}; set COMPANY.")

    data["_source_priority"] = data["_source_status"].map({"temporary": 1, "confirmed": 2})
    key_columns = ([COLS["company"]] if has_company else []) + [COLS["date"]]
    data = (
        data.sort_values(key_columns + ["_source_priority"])
        .drop_duplicates(key_columns, keep="last")
        .sort_values(key_columns)
        .reset_index(drop=True)
    )

    numeric_columns = [COLS[key] for key in ("equity", "cl", "ltd", "tl", "assets", "rf")]
    for column in numeric_columns:
        data[column] = pd.to_numeric(data[column], errors="coerce")

    # Financial-statement fields are not updated daily; forward-fill missing Temporary values from the latest known financial record.
    if has_company:
        data[BS_COLUMNS] = data.groupby(COLS["company"], dropna=False)[BS_COLUMNS].ffill()
    else:
        data[BS_COLUMNS] = data[BS_COLUMNS].ffill()

    if data[BS_COLUMNS].isna().any().any():
        raise ValueError("Financial-statement fields cannot be forward-filled; ensure the first history row has complete financial data.")
    return data.drop(columns="_source_priority")


def add_model_fields(frame: pd.DataFrame, delta: float = 0.5) -> pd.DataFrame:
    if not 0.0 <= delta <= 1.0:
        raise ValueError("delta must be between 0 and 1.")
    data = frame.copy()
    data[COLS["date"]] = _parse_dates(data[COLS["date"]])
    data = data.sort_values(COLS["date"]).reset_index(drop=True)
    data["Other_Liabilities(HKD)"] = data[COLS["tl"]] - data[COLS["cl"]] - data[COLS["ltd"]]
    data["Default_Point(HKD)"] = (
        data[COLS["cl"]] + 0.5 * data[COLS["ltd"]] + delta * data["Other_Liabilities(HKD)"]
    )
    data["Risk_Free_Decimal"] = data[COLS["rf"]].map(_rate_to_decimal)
    return data


def implied_asset_value(
    equity: float,
    debt: float,
    rate: float,
    sigma: float,
    maturity: float = 1.0,
) -> float:
    """Solve the market asset value V from E = V N(d1) - exp(-rT)L N(d2)."""
    equity, debt, rate, sigma, maturity = map(float, (equity, debt, rate, sigma, maturity))
    if equity <= 0 or debt <= 0 or sigma <= 0 or maturity <= 0:
        raise ValueError("equity, debt, sigma, and maturity must be positive.")
    sqrt_t = np.sqrt(maturity)

    def residual(asset_value: float) -> float:
        d1 = (
            np.log(asset_value / debt) + (rate + 0.5 * sigma**2) * maturity
        ) / (sigma * sqrt_t)
        d2 = d1 - sigma * sqrt_t
        option_value = asset_value * norm.cdf(d1) - debt * np.exp(-rate * maturity) * norm.cdf(d2)
        return option_value - equity

    lower = max(equity, 1e-9)
    upper = equity + debt * np.exp(-rate * maturity) + debt
    while residual(upper) < 0:
        upper *= 2.0
        if upper > 1e12 * max(1.0, debt):
            raise RuntimeError("Unable to establish a root bracket for market asset value V.")
    return float(brentq(residual, lower, upper, maxiter=200))


def _window_valid_rows(data: pd.DataFrame, as_of: pd.Timestamp, lookback_days: int) -> pd.DataFrame:
    start = as_of - pd.Timedelta(days=lookback_days)
    window = data[(data[COLS["date"]] > start) & (data[COLS["date"]] <= as_of)].copy()
    needed = REQUIRED_COLUMNS + ["Default_Point(HKD)", "Risk_Free_Decimal"]
    window = window.dropna(subset=needed)
    positive = (
        (window[COLS["equity"]] > 0)
        & (window[COLS["assets"]] > 0)
        & (window["Default_Point(HKD)"] > 0)
    )
    return window.loc[positive].reset_index(drop=True)


def estimate_sigma(
    history: pd.DataFrame,
    as_of,
    *,
    delta: float = 0.5,
    lookback_days: int = 365,
    min_obs: int = 50,
    trading_days_per_year: int = 250,
    sigma_bounds: tuple[float, float] = (0.005, 1.50),
) -> tuple[float, pd.DataFrame]:
    """Estimate asset volatility sigma using one-year rolling transformed-data MLE."""
    as_of = pd.Timestamp(as_of).normalize()
    data = add_model_fields(history, delta)
    window = _window_valid_rows(data, as_of, lookback_days)
    if len(window) < min_obs:
        raise ValueError(f"At least {min_obs} valid observations are required; only {len(window)} were provided")

    equity = window[COLS["equity"]].to_numpy(float)
    debt = window["Default_Point(HKD)"].to_numpy(float)
    book_assets = window[COLS["assets"]].to_numpy(float)
    rate = window["Risk_Free_Decimal"].to_numpy(float)
    h = np.full(len(window) - 1, 1.0 / trading_days_per_year)

    def negative_log_likelihood(sigma: float) -> float:
        try:
            market_assets = np.array(
                [implied_asset_value(equity[i], debt[i], rate[i], sigma) for i in range(len(window))]
            )
            d1 = (np.log(market_assets / debt) + rate + 0.5 * sigma**2) / sigma
            normal_d1 = np.clip(norm.cdf(d1), 1e-14, 1.0)
            transformed_return = np.log(
                (market_assets[1:] / book_assets[1:]) * (book_assets[:-1] / market_assets[:-1])
            )
            mu = 0.5 * sigma**2 + transformed_return.sum() / h.sum()
            residual = transformed_return - (mu - 0.5 * sigma**2) * h
            log_likelihood = (
                -0.5 * (len(window) - 1) * np.log(2 * np.pi)
                -0.5 * np.sum(np.log(sigma**2 * h))
                -np.sum(np.log(market_assets[1:] / book_assets[1:]))
                -np.sum(np.log(normal_d1[1:]))
                -0.5 / sigma**2 * np.sum((residual**2) / h)
            )
            return -float(log_likelihood) if np.isfinite(log_likelihood) else 1e100
        except (ValueError, RuntimeError, FloatingPointError):
            return 1e100

    optimum = minimize_scalar(
        negative_log_likelihood,
        bounds=sigma_bounds,
        method="bounded",
        options={"xatol": 1e-8},
    )
    if not optimum.success or not np.isfinite(optimum.fun):
        raise RuntimeError("Sigma optimization did not converge.")
    return float(optimum.x), window


def calculate_dtd(
    history: pd.DataFrame,
    as_of,
    *,
    delta: float = 0.5,
    lookback_days: int = 365,
    min_obs: int = 50,
    trading_days_per_year: int = 250,
) -> DTDResult:
    """Calculate the DTD using data dated on or before as_of only."""
    as_of = pd.Timestamp(as_of).normalize()
    data = add_model_fields(history, delta)
    available_dates = data[COLS["date"]]
    if not (available_dates == as_of).any():
        latest = available_dates[available_dates <= as_of].max()
        latest_text = "none" if pd.isna(latest) else latest.strftime("%Y-%m-%d")
        raise ValueError(f"Target date {as_of.date()} is absent from the input; the latest date on or before the target is {latest_text}")
    data = data[data[COLS["date"]] <= as_of].copy()
    row = data.loc[data[COLS["date"]] == as_of].iloc[-1]

    sigma, window = estimate_sigma(
        data,
        as_of,
        delta=delta,
        lookback_days=lookback_days,
        min_obs=min_obs,
        trading_days_per_year=trading_days_per_year,
    )
    market_assets = implied_asset_value(
        row[COLS["equity"]],
        row["Default_Point(HKD)"],
        row["Risk_Free_Decimal"],
        sigma,
    )
    # Under the supplied methodology, the CRI drift treatment causes this contribution to cancel.
    dtd = np.log(market_assets / row["Default_Point(HKD)"]) / sigma
    return DTDResult(
        date=as_of,
        dtd=float(dtd),
        sigma=float(sigma),
        asset_value=float(market_assets),
        default_point=float(row["Default_Point(HKD)"]),
        n_obs=len(window),
        delta=float(delta),
        source_status=str(row.get("_source_status", "unknown")),
    )


def run_dtd(
    confirmed_file: TableInput,
    as_of,
    temporary_file: Optional[TableInput] = None,
    *,
    company: Optional[object] = None,
    confirmed_sheet: str = "Input",
    temporary_sheet: str = "Input",
    delta: float = 0.5,
    lookback_days: int = 365,
    min_obs: int = 50,
    trading_days_per_year: int = 250,
    return_details: bool = False,
):
    """Primary entry point. Returns a float by default; return_details=True returns DTDResult."""
    clean_history = combine_confirmed_and_temporary(
        confirmed_file,
        temporary_file,
        confirmed_sheet=confirmed_sheet,
        temporary_sheet=temporary_sheet,
        company=company,
    )
    result = calculate_dtd(
        clean_history,
        as_of,
        delta=delta,
        lookback_days=lookback_days,
        min_obs=min_obs,
        trading_days_per_year=trading_days_per_year,
    )
    return result if return_details else result.dtd

## 3 Run and obtain one DTD value

The cell below displays diagnostics and leaves its final expression as a plain Python `float`. If only the number is needed, run:

```python
run_dtd(CONFIRMED_FILE, TARGET_DATE, TEMPORARY_FILE, company=COMPANY)
```

In [ ]:
details = run_dtd(
    confirmed_file=CONFIRMED_FILE,
    temporary_file=TEMPORARY_FILE,
    as_of=TARGET_DATE,
    company=COMPANY,
    confirmed_sheet=INPUT_SHEET,
    temporary_sheet=INPUT_SHEET,
    return_details=True,
)

display(pd.Series(details.as_dict(), name="value"))
dtd_number = float(details.dtd)
dtd_number